# ShopDesk, Lab 1: From a Weak Prompt to a Structured One

A beginner-friendly notebook built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**. Same ShopDesk support agent
as the other modules; this time we take one weak prompt and refine it, step by step,
into a structured one, then **measure** the quality difference across versions.

## The real-world scenario

ShopDesk gets a customer message and has to answer it correctly and consistently. A
vague, one-line prompt ("Answer the customer.") drifts: sometimes prose, sometimes a
list, sometimes the wrong refund call because the model never saw the order facts. That
is impossible to trust with real money.

The question this lab answers: **which prompting techniques turn that weak prompt into
a reliable one, and how much does each one actually help?**

## Objectives

- Refine one prompt across six versions, adding a single technique each time: **role
  prompting**, **clear instructions + an explicit output format**, **XML-tag
  structuring**, **few-shot examples**, and **chain-of-thought / extended thinking**.
- **Score** every version against the same rubric so the improvement is a number, not
  a feeling.

## What you'll observe

- Each version prints the exact prompt it sends, so you can see the technique added.
- The scorer grades every output out of 5: valid JSON, all required fields, correct
  order id, correct refund decision, and a grounded reply.
- The score climbs as structure and grounding are added; the weak prompt scores low,
  the structured one scores high.

## How to run

Run top to bottom. The prompt-building and scoring cells are pure Python and run
anywhere. To run the **live** model cells, paste a real key into **Setup 2/3** and
re-run from the top; otherwise they skip cleanly and the scorer runs on sample outputs
so you still see the measurement.

## 0. Setup

**This cell:** installs the packages. This lab uses only the **base Anthropic
SDK** (no Agent SDK, no Node.js), because prompt design lives entirely in the
`system` and `user` fields of one Messages API call. We install once so every cell
below can just import.

In [ ]:
# ===== SETUP 1/4 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports what we need, pins the model, and sets a `RUN_LIVE`
switch so live calls fire only when a real key is present. We build the switch now so
every live cell can guard itself and still run offline.

In [ ]:
# ===== SETUP 2/4 - imports, the model, and a live/offline switch =====
import os                                      # read the API key from the environment
import json                                    # parse and grade JSON outputs
import anthropic                               # the base Anthropic SDK (synchronous)

try:                                           # load a .env file if present (keeps keys out of code)
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                              #   read .env into environment variables
except Exception:                              # python-dotenv not installed? that is fine
    pass                                       #   the key can be set another way

MODEL = "claude-sonnet-4-6"                     # the Sonnet model every cell will call

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]          # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world**: the orders, who owns them,
and the status-code map. We define the data on its own first because both the prompts
(which need the order facts) and the scorer (which checks the answer) read from it.

In [ ]:
# ===== SETUP 3/4 - the shared ShopDesk data =====
ORDERS = {                                      # our tiny order book
    "A1": {"status": 2, "refundable": True},    # fresh  -> within the 30-day window
    "A2": {"status": 3, "refundable": False},   # old    -> past it
}
CUSTOMERS = {                                   # who owns each order
    "A1": {"name": "Ravi"},
    "A2": {"name": "Meera"},
}
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # code -> human word
print("orders:", list(ORDERS), "| refund window: 30 days")   # quick sanity check

**This cell:** defines the one **test case**: the customer message every version
will answer, and the **ground truth** (the correct answer). We separate this from the
data because it is what the scorer grades against; changing the test later means editing
only this cell.

In [ ]:
# ===== SETUP 4/4 - the test ticket and the correct answer =====
TICKET = "Hi, this is Meera. I want to return order A2 and get my money back. It arrived a while ago but I changed my mind."   # the message we test

TRUTH = {                                       # the correct answer we grade against
    "order_id": "A2",                           #   which order
    "intent": "refund",                         #   what the customer wants
    "decision": "refuse",                       #   A2 is past the 30-day window
    "reason": "past 30-day window",             #   why
}
print("testing on order:", TRUTH["order_id"], "| correct decision:", TRUTH["decision"])

**This cell:** defines `ask()`, the one function that talks to Claude. Every
prompt version calls it with a different `system` and `user`, so the *only* thing that
changes between versions is the prompt itself, never the plumbing. It also accepts a
`use_thinking` flag for the last version.

In [ ]:
# ===== the single call every version shares =====
def ask(system, user, use_thinking=False):                  # send one prompt, return the text answer
    client = anthropic.Anthropic()                          # reads ANTHROPIC_API_KEY from the environment
    kwargs = dict(model=MODEL, max_tokens=2000,             # max_tokens must exceed any thinking budget below
                  messages=[{"role": "user", "content": user}])   # the user turn
    if system:                                              # only attach a system prompt when we actually have one
        kwargs["system"] = system                           #   v0 sends none; v1+ send one
    if use_thinking:                                        # v5: give the model a reasoning scratchpad
        kwargs["thinking"] = {"type": "enabled", "budget_tokens": 1024}   # 1024 is the minimum budget
    r = client.messages.create(**kwargs)                    # one Messages API call
    return "".join(b.text for b in r.content if b.type == "text")   # join only the text blocks (skip thinking)

---

### 🎯 Lab objective - refine one prompt, measure each step

**What you build:** six versions of the same ShopDesk prompt, each adding one technique,
plus a rubric that scores every output.

**Why it helps you build real solutions:** prompting is not magic words; it is a set of
levers (role, format, structure, examples, reasoning) you add on purpose. Scoring turns
"this feels better" into a number you can defend.

**How you'll see it:** the printed prompts grow more structured, and the scores climb
from the weak version to the structured one.

**This cell:** version 0, the **weak** prompt. No role, no instructions, no
format, no order facts: just the raw message. We build it first as the baseline every
later version has to beat.

In [ ]:
# ===== v0 - the weak baseline =====
VERSIONS = {}                                   # name -> (system, user, use_thinking)

SYS_V0 = ""                                     # no system prompt at all
USER_V0 = TICKET                                # just hand over the raw customer message
VERSIONS["v0 weak"] = (SYS_V0, USER_V0, False)  # register the baseline
print("USER:", USER_V0)                         # show exactly what we send

**This cell:** version 1 adds **role prompting**. We give the model an identity,
a tone, and the one policy rule (the 30-day window). Role prompting steadies behaviour,
but note we still have not told it the *specific* facts about order A2.

In [ ]:
# ===== v1 - role prompting =====
SYS_V1 = (                                      # a system prompt that defines who the model is
    "You are ShopDesk, a calm, concise e-commerce support agent. "   # role + tone
    "Refunds are allowed only within 30 days of delivery; refuse anything past it."   # the one rule
)
USER_V1 = TICKET                                # same message, unchanged
VERSIONS["v1 role"] = (SYS_V1, USER_V1, False)  # register it
print(SYS_V1)                                   # show the new system prompt

**This cell:** version 2, part one, the **system prompt**. We add **clear
instructions** (a numbered procedure) and an **explicit output format** (exact JSON
keys). We build the system prompt on its own so you can see that the format demand is
what makes the output parseable later.

In [ ]:
# ===== v2 (a) - the system prompt: clear steps + explicit JSON format =====
SYS_V2 = (                                      # role + a numbered procedure + an exact output shape
    "You are ShopDesk, a calm, concise e-commerce support agent.\n"
    "Follow these steps:\n"
    "1. Identify the order id and what the customer wants.\n"
    "2. Apply the rule: refunds only within 30 days of delivery.\n"
    "3. Decide: approve or refuse.\n"
    "Respond with ONLY a JSON object, no prose, with keys: "
    "order_id, intent, decision, reason, reply."
)
print(SYS_V2)                                   # show the instructions we will send

**This cell:** version 2, part two, the **grounded user message**. Clear
instructions include giving the model its data, so here we attach the order facts.
Without them no prompt can know A2 is past the window; this is where grounding enters,
and we register the version so the driver can run it.

In [ ]:
# ===== v2 (b) - the user message: the ticket PLUS the order facts =====
facts_a2 = ORDERS["A2"]                          # the specific facts for this order
USER_V2 = (                                      # the message PLUS the facts it needs
    TICKET + "\n\n"
    "Known facts: order A2 status=" + STATUS_NAMES[facts_a2["status"]] +
    ", refundable=" + str(facts_a2["refundable"]) + "."
)
VERSIONS["v2 format"] = (SYS_V2, USER_V2, False)  # register this version
print(USER_V2)                                   # show the grounded message

**This cell:** version 3 adds **XML-tag structuring**, part one: the **system
prompt**. We ask the model to read tagged inputs and reply inside a `<result>` tag. Tags
cut ambiguity between instruction and data, which matters more as inputs get longer.

In [ ]:
# ===== v3 (a) - the system prompt asks for tagged input and output =====
SYS_V3 = (                                       # same job, now asking for tagged, structured output
    "You are ShopDesk, a calm, concise e-commerce support agent.\n"
    "Refunds are allowed only within 30 days of delivery.\n"
    "Read <customer_message> and <order_facts>, then return ONLY a <result> tag "
    "containing a JSON object with keys: order_id, intent, decision, reason, reply."
)
print(SYS_V3)                                    # show the tag-aware instructions

**This cell:** part two, the **tagged user message**. We wrap the same customer
message and order facts in `<customer_message>` and `<order_facts>` tags, then register
the version. The content is identical to v2; only the structuring is new.

In [ ]:
# ===== v3 (b) - wrap the input in tags and register the version =====
USER_V3 = (                                      # the same content, now clearly delimited
    "<customer_message>" + TICKET + "</customer_message>\n"
    "<order_facts>order_id=A2 status=" + STATUS_NAMES[ORDERS['A2']['status']] +
    " refundable=" + str(ORDERS['A2']['refundable']) + "</order_facts>"
)
VERSIONS["v3 xml"] = (SYS_V3, USER_V3, False)    # register this version
print(USER_V3)                                   # show the tagged input

**This cell:** version 4 adds **few-shot examples**, part one: **write the
examples**. We show two worked cases (a past-window refuse and an in-window approve)
using *different* orders, so the model copies the shape and the reasoning without copying
the answer. Examples teach format and edge-case handling at once.

In [ ]:
# ===== v4 (a) - write the two demonstration examples =====
EXAMPLES = (                                      # two input -> output demonstrations
    "<examples>\n"
    "<example>\n"
    "  <order_facts>order_id=Z9 status=delivered refundable=false</order_facts>\n"
    '  <result>{"order_id":"Z9","intent":"refund","decision":"refuse",'
    '"reason":"past 30-day window","reply":"Order Z9 is past the 30-day window, so a refund is not possible."}</result>\n'
    "</example>\n"
    "<example>\n"
    "  <order_facts>order_id=Z1 status=shipped refundable=true</order_facts>\n"
    '  <result>{"order_id":"Z1","intent":"refund","decision":"approve",'
    '"reason":"within 30-day window","reply":"Order Z1 is eligible, so your refund is approved."}</result>\n'
    "</example>\n"
    "</examples>"
)
print(EXAMPLES)                                   # show the two examples we will teach from

**This cell:** part two, **assemble** version 4. We append the examples to v3's
instructions and reuse v3's input unchanged, so the only difference from v3 is the
demonstrations. Keeping the assembly separate makes that one change obvious.

In [ ]:
# ===== v4 (b) - assemble and register the few-shot version =====
SYS_V4 = SYS_V3 + "\n" + EXAMPLES                # reuse v3's instructions, then append the examples
USER_V4 = USER_V3                                 # the input is identical to v3
VERSIONS["v4 fewshot"] = (SYS_V4, USER_V4, False) # register this version
print("v4 system prompt length:", len(SYS_V4), "chars")   # confirm the examples were added

**This cell:** version 5 adds **chain-of-thought via extended thinking**. The
prompt is the same as v4, but we flip `use_thinking=True`, giving the model a private
reasoning scratchpad before it answers. On the newer models the adaptive form
`thinking={"type":"adaptive"}` is the modern alternative; we use the explicit budget
here because it maps cleanly to the idea of "how much room to reason."

In [ ]:
# ===== v5 - chain-of-thought / extended thinking =====
SYS_V5 = SYS_V4 + "\nThink through the rule before you answer."   # nudge it to reason first
USER_V5 = USER_V4                                 # same input as v4
VERSIONS["v5 thinking"] = (SYS_V5, USER_V5, True) # note the True: this version turns thinking ON
print("thinking budget:", 1024, "tokens (set in ask())")   # a reminder of where the budget lives

**This cell:** the first half of the scorer, `parse_json()`. Models sometimes
wrap JSON in a ```json fence or a <result> tag, so we need one robust reader that digs the
JSON object out of whatever text comes back. We build it on its own because every grade
depends on parsing succeeding first.

In [ ]:
# ===== scorer, part 1: robustly read JSON out of the model's text =====
def parse_json(text):                             # pull a JSON object out of possibly-messy text
    t = text.strip()                              #   trim whitespace
    if "```" in t:                                #   strip a ```json ... ``` fence if present
        t = t.split("```")[1].replace("json", "", 1)   #     take the fenced middle
    if "<result>" in t:                           #   strip a <result> ... </result> tag if present
        t = t.split("<result>")[1].split("</result>")[0]   #     take the tagged middle
    a, b = t.find("{"), t.rfind("}")              #   locate the outermost braces
    try:                                          #
        return json.loads(t[a:b+1])               #   parse just the JSON span
    except Exception:                             #   not parseable?
        return None                               #     signal failure

**This cell:** the second half, `score()` and its rubric constants. It runs the
five checks (valid JSON, all fields, right order, right decision, grounded reply) and
returns a number out of 5 plus a per-check breakdown. Because scoring is code, the same
rubric judges every version fairly.

In [ ]:
# ===== scorer, part 2: grade a parsed output out of 5 =====
REQUIRED = ["order_id", "intent", "decision", "reason", "reply"]   # fields a good answer must have
REFUSALS = {"refuse", "deny", "declined", "no", "reject"}          # words that count as a refusal

def score(text):                                  # returns (points_out_of_5, checklist)
    checks = {"json": False, "fields": False, "order": False, "decision": False, "grounded": False}
    data = parse_json(text)                       #   try to read structured output
    checks["json"] = data is not None             #   1) did it parse as JSON?
    if data:                                       #   only grade the rest if we have a dict
        checks["fields"] = all(k in data for k in REQUIRED)          # 2) all required fields present?
        checks["order"] = str(data.get("order_id", "")).upper() == TRUTH["order_id"]   # 3) right order?
        dec = str(data.get("decision", "")).lower()                 #   the decision, lowercased
        checks["decision"] = any(w in dec for w in REFUSALS)        # 4) correctly a refusal?
        checks["grounded"] = TRUTH["order_id"] in str(data.get("reply", ""))   # 5) reply names the order?
    return sum(checks.values()), checks           #   total points and the per-check breakdown

**This cell:** three **sample outputs** (weak, partial, strong) so the scorer has
something to grade even without an API key. When live, we grade the real model outputs
instead; these samples exist only to show the measurement offline.

In [ ]:
# ===== sample outputs, used only when running offline =====
SAMPLES = {
    "weak prose":   "Sure, I can help you return that. Let me look into it for you!",   # no structure at all
    "partial json": '{"order_id":"A2","decision":"approve"}',                            # JSON but missing fields and wrong call
    "strong json":  '{"order_id":"A2","intent":"refund","decision":"refuse",'            # correct, complete, grounded
                    '"reason":"past 30-day window","reply":"Order A2 is past the 30-day window, so a refund is not possible."}',
}
print("offline samples ready:", list(SAMPLES))    # confirm they exist

**This cell:** a tiny `show()` helper that prints one graded row (label, score,
and which checks passed). We define it on its own so the run loop in the next cell stays
short and readable.

In [ ]:
# ===== the row printer =====
def show(label, pts, checks):                     # print one graded row compactly
    passed = [k for k, v in checks.items() if v]  #   list the checks that passed
    print(f"{label:<14} {pts}/5   passed: {passed}")   # aligned label + score + passes

**This cell:** runs the whole experiment. Live, it sends every version to Claude
and scores the result; offline, it scores the three sample outputs. Either way you get a
table where the score rises as we add technique and grounding.

In [ ]:
# ===== run every version (or the samples) and compare the scores =====
if RUN_LIVE:                                       # real key present: grade real outputs
    for name, (system, user, thinking) in VERSIONS.items():   # walk the six versions in order
        out = ask(system, user, thinking)          #   call Claude with this version's prompt
        pts, checks = score(out)                    #   grade the output
        show(name, pts, checks)                     #   print the graded row
else:                                               # no key: grade the sample outputs instead
    print("[offline] grading sample outputs to show the rubric:")
    for label, sample in SAMPLES.items():           #   walk weak -> partial -> strong
        pts, checks = score(sample)                 #   grade each sample
        show(label, pts, checks)                    #   print the graded row

| anti-pattern | what to do instead |
|---|---|
| pile on flowery adjectives and hope | add one lever at a time: role, format, structure, examples, reasoning |
| ask for JSON but never give the facts | ground the prompt with the order data the answer depends on |
| judge prompts by eye | score every version against a fixed rubric so the gain is a number |
| enable extended thinking everywhere | reserve it for the step where reasoning actually changes the answer |

**Lesson:** a prompt is built, not guessed. **Role** sets behaviour, **clear
instructions plus a format** make the output parseable, **XML tags** separate
instruction from data, **few-shot examples** teach shape and edge cases, and
**extended thinking** buys reasoning where it matters. Grounding (giving the model the
facts) is what makes any of it *correct*. The rubric is how you prove each step helped.

---

## Recap - one prompt, five levers, one number

| Version | Technique added | What the score shows |
|---|---|---|
| v0 | (weak baseline) | low: unstructured, ungrounded |
| v1 | role prompting | steadier tone, still ungrounded |
| v2 | clear instructions + format + grounding | jumps once output is JSON and facts are present |
| v3 | XML-tag structuring | cleaner separation of instruction and data |
| v4 | few-shot examples | shape and edge cases locked in |
| v5 | chain-of-thought / extended thinking | reasoning applied before the answer |

One principle ties them together: **the model reasons; your prompt sets the rails, and
your rubric keeps you honest.** To run live, paste a real key into **Setup 2/3** and
re-run from the top. Then try it: add a second ticket for order A1 (a valid refund) and
watch the same versions handle it.